In [1]:
import pandas as pd

df = pd.read_csv('/content/output.csv')
print(df.head())

          filename   type
0     two_col_1718   line
1     two_col_3725  v_bar
2     two_col_2837  h_bar
3    two_col_62298  v_bar
4  multi_col_20558  v_bar


In [2]:
for col in df.columns:
  print(f"Column '{col}': {df[col].nunique()} unique values")

Column 'filename': 18317 unique values
Column 'type': 4 unique values


In [5]:
print(df['type'].unique())

['line' 'v_bar' 'h_bar' 'pie']


In [26]:
dfs = {}
filenames = ['/content/train_split_files.csv', '/content/val_split_files.csv', '/content/test_split_files.csv']

for filename in filenames:
  df_name = filename.split('/')[-1].split('.')[0] # Extract name like 'test_split_files'
  dfs[df_name] = pd.read_csv(filename)
  print(f"\nAnalyzing {filename}:")
  print(dfs[df_name].head())

  for col in dfs[df_name].columns:
    print(f"Column '{col}': {dfs[df_name][col].nunique()} unique values")

  if 'type' in dfs[df_name].columns:
      print(dfs[df_name]['type'].unique())

      type_counts = dfs[df_name]['type'].value_counts()
      type_percentages = (type_counts / len(dfs[df_name])) * 100

      for index, value in type_counts.items():
        percentage = type_percentages[index]
        print(f"{index}: {value} ({percentage:.2f}%)")
  else:
      print("No 'type' column found in this file.")

  print(f"Total records in {filename}: {len(dfs[df_name])}")


Analyzing /content/train_split_files.csv:
         filename   type
0   two_col_43791  v_bar
1  84985912004897  h_bar
2   two_col_42338  h_bar
3            6390   line
4   two_col_20849  v_bar
Column 'filename': 12821 unique values
Column 'type': 4 unique values
['v_bar' 'h_bar' 'line' 'pie']
v_bar: 7110 (55.46%)
h_bar: 3796 (29.61%)
line: 1536 (11.98%)
pie: 379 (2.96%)
Total records in /content/train_split_files.csv: 12821

Analyzing /content/val_split_files.csv:
          filename   type
0   08152707004883  h_bar
1   two_col_104699  h_bar
2  multi_col_20260  h_bar
3            10395   line
4      two_col_752   line
Column 'filename': 2748 unique values
Column 'type': 4 unique values
['h_bar' 'line' 'v_bar' 'pie']
v_bar: 1524 (55.46%)
h_bar: 813 (29.59%)
line: 330 (12.01%)
pie: 81 (2.95%)
Total records in /content/val_split_files.csv: 2748

Analyzing /content/test_split_files.csv:
          filename   type
0  multi_col_40425  v_bar
1    two_col_61458  h_bar
2    two_col_20433  v_bar
3

In [22]:
files_to_process = ['/content/input_train_annot_t5.csv', '/content/input_val_annot_t5.csv', '/content/input_test_annot_t5.csv']

for file_path in files_to_process:
  print(f"\nProcessing file: {file_path}")
  try:
    df_input = pd.read_csv(file_path)

    # Count the total values in the 'input' column
    total_input_values = df_input['input'].count()

    # Print the total count
    print(f"Total values in the 'input' column of '{file_path}': {total_input_values}")

  except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
  except Exception as e:
    print(f"An error occurred while processing {file_path}: {e}")


Processing file: /content/input_train_annot_t5.csv
Total values in the 'input' column of '/content/input_train_annot_t5.csv': 19789

Processing file: /content/input_val_annot_t5.csv
Total values in the 'input' column of '/content/input_val_annot_t5.csv': 4276

Processing file: /content/input_test_annot_t5.csv
Total values in the 'input' column of '/content/input_test_annot_t5.csv': 4234


In [33]:
import json

file1 = '/content/augmented.json'
file2 = '/content/human.json'
total_items_file1 = 0
total_items_file2 = 0

with open(file1, 'r') as f:
    data1 = json.load(f)
    if isinstance(data1, list):
        total_items_file1 = len(data1)
    elif isinstance(data1, dict):
        total_items_file1 = len(data1)

with open(file2, 'r') as f:
    data2 = json.load(f)
    if isinstance(data2, list):
        total_items_file2 = len(data2)
    elif isinstance(data2, dict):
        total_items_file2 = len(data2)

print(f"Total items in {file1}: {total_items_file1}")
print(f"Total items in {file2}: {total_items_file2}")
print(f"Total items in both files: {total_items_file1 + total_items_file2}")

Total items in /content/augmented.json: 20901
Total items in /content/human.json: 7398
Total items in both files: 28299


In [36]:
json_files = {
    'train_augmented': '/content/train_augmented.json',
    'train_human': '/content/train_human.json',
    'val_augmented': '/content/val_augmented.json',
    'val_human': '/content/val_human.json',
    'test_augmented': '/content/test_augmented.json',
    'test_human': '/content/test_human.json'
}

total_all_files = 0
totals_by_prefix = {} # To store totals for train, val, test prefixes

for name, file_path in json_files.items():
  try:
    with open(file_path, 'r') as f:
      data = json.load(f)
      if isinstance(data, list):
        item_count = len(data)
      elif isinstance(data, dict):
        item_count = len(data)
      else:
        item_count = 0
      print(f"Total items in {file_path}: {item_count}")
      total_all_files += item_count

      # Aggregate totals by prefix (train, val, test)
      prefix = name.split('_')[0]
      totals_by_prefix[prefix] = totals_by_prefix.get(prefix, 0) + item_count

  except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
  except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from {file_path}")
  except Exception as e:
    print(f"An error occurred while processing {file_path}: {e}")

print(f"\nTotal items in all files combined: {total_all_files}")

# Print total for each prefix
for prefix, total in totals_by_prefix.items():
    print(f"Total items for {prefix} files: {total}")

Total items in /content/train_augmented.json: 14591
Total items in /content/train_human.json: 5198
Total items in /content/val_augmented.json: 3208
Total items in /content/val_human.json: 1068
Total items in /content/test_augmented.json: 3102
Total items in /content/test_human.json: 1132

Total items in all files combined: 28299
Total items for train files: 19789
Total items for val files: 4276
Total items for test files: 4234
